In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv("../../data/product_type_2.csv", header=[0,1])
df

process                                                     \
           id product_type shot velocity_1 velocity_2 velocity_3   
0     4207011            2   11      0.156      0.166      0.192   
1     4208012            2   12      0.157      0.166      0.204   
2     4209013            2   13      0.156      0.170      0.204   
3     4210014            2   14      0.154      0.170      0.202   
4     4211015            2   15      0.146      0.160      0.198   
...       ...          ...  ...        ...        ...        ...   
1959  7525657            2  657      0.144      0.173      0.200   
1960  7527658            2  658      0.144      0.173      0.200   
1961  7529659            2  659      0.150      0.166      0.210   
1962  7531660            2  660      0.144      0.174      0.206   
1963  7533661            2  661      0.147      0.174      0.204   

                                                                        ...  \
     high_velocity cylinder_pressure rapid_rise_time biscuit_thickness  ...   
0            2.723               265           0.012                20  ...   
1            2.730               264           0.014                19  ...   
2            2.715               265           0.012                18  ...   
3            2.717               264           0.011                20  ...   
4            2.684               264           0.012                20  ...   
...            ...               ...             ...               ...  ...   
1959         2.536               264           0.012                17  ...   
1960         2.536               264           0.012                17  ...   
1961         2.492               265           0.011                17  ...   
1962         2.514               264           0.011                16  ...   
1963         2.532               265           0.012                18  ...   

     defects                                                          \
     stain_2 dent_2 deformation_2 contamination_2 impurity_2 crack_2   
0          0      0             0               0          0       0   
1          0      0             0               0          0       0   
2          0      0             0               0          0       0   
3          0      0             0               0          0       0   
4          0      0             0               0          0       0   
...      ...    ...           ...             ...        ...     ...   
1959       0      0             0               0          0       0   
1960       0      0             0               0          0       0   
1961       0      0             0               0          0       0   
1962       0      0             0               0          0       0   
1963       0      0             0               0          0       0   

                                          defect_flag  
     scratch_2 buring_mark_2 inclusions_2   is_defect  
0            0             0            0           0  
1            0             0            0           0  
2            0             0            0           0  
3            0             0            0           0  
4            0             0            0           0  
...        ...           ...          ...         ...  
1959         0             0            0           0  
1960         0             0            0           0  
1961         0             0            0           0  
1962         0             0            0           1  
1963         0             0            0           1  

[1964 rows x 58 columns]

In [ ]:
### 1. 불필요한 id, product_type, shot 컬럼 제거
df = df.drop(columns=[('process', 'id'), ('process', 'shot'), ('process', 'product_type')])

In [ ]:
### 2. 값이 전부 0인 defects 컬럼 drop
# Defects 하위 컬럼들
defect_cols = df['defects'].columns

# 값이 전부 0인 Defects 컬럼 찾기
zero_defects = [
    col for col in defect_cols
    if df[('defects', col)].sum() == 0
]

# 결과 출력
print("삭제될 defects 컬럼:")
print(zero_defects)

print("\n삭제될 컬럼 수:", len(zero_defects))
print("삭제 전 defects 컬럼 수:", len(defect_cols))

# 실제 삭제
df = df.drop(columns=[('defects', col) for col in zero_defects])

print("삭제 후 defects 컬럼 수:", len(df['defects'].columns))

In [ ]:
df['defects'].sum().sort_values(ascending=False)

In [ ]:
# 불량 범주 정의 (suffix _1, _2 자동 처리)
defect_groups = {
    "표면": [
        "stain_1",
        "dent_1",
        "dent_2",
        "scratch_1",
        "buring_mark_1"
    ],

    "구조": [
        "short_shot_1",
        "short_shot_2",
        "blow_hole_1",
        "blow_hole_2",
        "bubble_1",
        "crack_1",
        "crack_2"
    ],

    "이물질": [
        "contamination_1",
        "contamination_2",
        "impurity_1",
        "impurity_2",
        "inclusions_2"
    ]
}

In [ ]:
# 입력 변수
X = df['Process'].copy()

# 혹시 결측이 있으면 일단 중앙값으로 채우기
X = X.fillna(X.median())

# Stage 1 타겟: 하나라도 불량이면 1
y_binary = (y_group.sum(axis=1) > 0).astype(int)

print("정상/불량 분포")
print(y_binary.value_counts())
print(y_binary.value_counts(normalize=True))

In [ ]:
X_train, X_test, y_train_bin, y_test_bin, y_train_group, y_test_group = train_test_split(
    X, y_binary, y_group,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print(X_train.shape, X_test.shape)

In [ ]:
# SMOTE 적용
smote_stage1 = SMOTE(random_state=42)
X_train_sm1, y_train_bin_sm = smote_stage1.fit_resample(X_train, y_train_bin)

print("SMOTE 후 Stage1 분포")
print(pd.Series(y_train_bin_sm).value_counts())

# Stage 1 모델
stage1_model = XGBClassifier(
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

stage1_model.fit(X_train_sm1, y_train_bin_sm)

In [ ]:
# 불량 확률
stage1_proba = stage1_model.predict_proba(X_test)[:, 1]

# threshold 조정
threshold = 0.3
stage1_pred = (stage1_proba >= threshold).astype(int)

print("=== Stage 1: 불량 여부 분류 ===")
print(classification_report(y_test_bin, stage1_pred, zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test_bin, stage1_pred))

In [ ]:
# 단일 불량만 있는 행만 사용
single_defect_mask = (y_group.sum(axis=1) == 1)

# 대표 불량 유형
y_type = y_group.idxmax(axis=1)

# Stage 2 전체 후보
X_type = X[single_defect_mask]
y_type = y_type[single_defect_mask]

print("Stage 2 학습 후보 분포")
print(y_type.value_counts())
print(y_type.value_counts(normalize=True))

In [ ]:
# Stage 1 split 기준으로 Stage 2 train/test 구성
train_idx = X_train.index
test_idx = X_test.index

stage2_train_mask = single_defect_mask.loc[train_idx]
stage2_test_mask = single_defect_mask.loc[test_idx]

X_train_stage2 = X.loc[train_idx][stage2_train_mask]
y_train_stage2 = y_type.loc[train_idx][stage2_train_mask]

X_test_stage2 = X.loc[test_idx][stage2_test_mask]
y_test_stage2 = y_type.loc[test_idx][stage2_test_mask]

print("Stage 2 train shape:", X_train_stage2.shape)
print("Stage 2 test shape:", X_test_stage2.shape)
print("\nStage 2 train class 분포")
print(y_train_stage2.value_counts())

In [ ]:
# 이물질 데이터가 매우 적을 수 있으므로 k_neighbors를 작게
smote_stage2 = SMOTE(random_state=42, k_neighbors=2)

X_train_sm2, y_train_stage2_sm = smote_stage2.fit_resample(X_train_stage2, y_train_stage2)

print("SMOTE 후 Stage2 분포")
print(pd.Series(y_train_stage2_sm).value_counts())

stage2_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    n_estimators=200,
    max_depth=5,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

stage2_model.fit(X_train_sm2, y_train_stage2_sm)

In [ ]:
stage2_pred = stage2_model.predict(X_test_stage2)

print("=== Stage 2: 불량 종류 분류 ===")
print(classification_report(y_test_stage2, stage2_pred, zero_division=0))

In [ ]:
# Stage 1 확률 / 예측
stage1_proba_test = stage1_model.predict_proba(X_test)[:, 1]
stage1_pred_test = (stage1_proba_test >= threshold).astype(int)

# 기본 결과 테이블
final_pred = pd.DataFrame(index=X_test.index)
final_pred["불량확률"] = stage1_proba_test
final_pred["불량여부_예측"] = stage1_pred_test
final_pred["예측_불량종류"] = "정상"

# Stage 1에서 불량으로 예측된 샘플만 Stage 2 분류
pred_defect_idx = final_pred.index[final_pred["불량여부_예측"] == 1]

if len(pred_defect_idx) > 0:
    stage2_pred_for_defects = stage2_model.predict(X_test.loc[pred_defect_idx])
    final_pred.loc[pred_defect_idx, "예측_불량종류"] = stage2_pred_for_defects

final_pred.head(20)

In [ ]:
actual_result = pd.DataFrame(index=X_test.index)
actual_result["실제_불량여부"] = y_test_bin.values
actual_result["실제_불량종류"] = "정상"

# 실제 단일 불량 범주인 경우만 실제 종류 표시
single_test_idx = y_test_group.index[(y_test_group.sum(axis=1) == 1)]
actual_result.loc[single_test_idx, "실제_불량종류"] = y_test_group.loc[single_test_idx].idxmax(axis=1)

result_compare = final_pred.join(actual_result)
result_compare.head(20)